# Advanced Problems with Solutions: Variable Re-Assignment

This notebook explores **variable re-assignment**, **object identity**, **mutability**, **aliasing**, and related Python behaviors.

## Learning goals
- Understand that variables are **names bound to objects**, not boxes that store values.
- Use `id()` to reason about object identity.
- Distinguish **re-assignment** from **in-place mutation**.
- Recognize when two variables refer to the **same object**.
- Avoid common mistakes involving aliasing and mutable objects.

> **Important note:** The exact numeric values returned by `id()` are implementation-dependent and may differ across runs and environments. Focus on whether identities are the **same or different**, not on specific numbers.


## Quick refresher

When you write:

```python
a = 10
```

the name `a` becomes bound to an integer object. If later you write:

```python
a = 15
```

then `a` is re-bound to a different object. This is **re-assignment**.

For immutable types like `int`, `str`, and `tuple`, operations that appear to "change" a value usually create a **new object** instead.


## Warm-up example


In [1]:
a = 10
print('a =', a, '| id(a) =', hex(id(a)))

a = 15
print('a =', a, '| id(a) =', hex(id(a)))

a = 5
print('a =', a, '| id(a) =', hex(id(a)))

a = a + 1
print('a =', a, '| id(a) =', hex(id(a)))


a = 10 | id(a) = 0x7ffc8aac74c8
a = 15 | id(a) = 0x7ffc8aac7568
a = 5 | id(a) = 0x7ffc8aac7428
a = 6 | id(a) = 0x7ffc8aac7448


You will often observe that the identity changes after each re-assignment.

Now compare:


In [2]:
a = 10
b = 10
print('id(a) =', hex(id(a)))
print('id(b) =', hex(id(b)))
print('a is b ->', a is b)


id(a) = 0x7ffc8aac74c8
id(b) = 0x7ffc8aac74c8
a is b -> True


In many Python implementations, `a` and `b` may point to the same integer object for small integers. This is related to **object reuse / interning optimizations**.

That does **not** mean variables are the same thing; it means both names currently refer to the same object.


---
# Problem 1 - Predict identity changes for immutable objects

Consider the code below. Before running it, predict which `id(...)` values will match and which will differ. Then verify your prediction.


In [3]:
x = 100
id1 = id(x)

x = x + 0
id2 = id(x)

x = x + 1
id3 = id(x)

print(id1, id2, id3)
print('id1 == id2:', id1 == id2)
print('id2 == id3:', id2 == id3)


140722635046920 140722635046920 140722635046952
id1 == id2: True
id2 == id3: False


## Solution 1

- `x = x + 0` computes a value equal to the previous one. Depending on implementation details, Python may reuse the same integer object here, so `id1 == id2` may be `True`.
- `x = x + 1` changes the value, so `x` must refer to an object representing a different integer value; typically `id2 != id3`.

**Key idea:** For immutable objects, any apparent "change" results in re-binding the variable name to an object, often a different one.


---
# Problem 2 - `is` versus `==`

Explain the difference between identity and equality, then test the following examples.


In [4]:
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print('a == b ->', a == b)
print('a is b ->', a is b)
print('a == c ->', a == c)
print('a is c ->', a is c)


a == b -> True
a is b -> False
a == c -> True
a is c -> True


## Solution 2

- `==` checks whether two objects have the **same value/content**.
- `is` checks whether two variables refer to the **same object in memory**.

So here:
- `a == b` is `True` because the lists contain the same elements.
- `a is b` is `False` because they are two separate list objects.
- `a is c` is `True` because `c = a` makes `c` another name for the same object.

**Best practice:** Use `==` for value comparison. Use `is` mainly for identity-sensitive checks, such as `x is None`.


---
# Problem 3 - Re-assignment versus mutation

Study the difference between mutating a list and re-assigning a variable.


In [5]:
nums = [1, 2, 3]
alias = nums

print('Before mutation:')
print('nums =', nums, '| id(nums) =', id(nums))
print('alias =', alias, '| id(alias) =', id(alias))

nums.append(4)
print('\nAfter nums.append(4):')
print('nums =', nums, '| id(nums) =', id(nums))
print('alias =', alias, '| id(alias) =', id(alias))

nums = nums + [5]
print('\nAfter nums = nums + [5]:')
print('nums =', nums, '| id(nums) =', id(nums))
print('alias =', alias, '| id(alias) =', id(alias))


Before mutation:
nums = [1, 2, 3] | id(nums) = 1579329237440
alias = [1, 2, 3] | id(alias) = 1579329237440

After nums.append(4):
nums = [1, 2, 3, 4] | id(nums) = 1579329237440
alias = [1, 2, 3, 4] | id(alias) = 1579329237440

After nums = nums + [5]:
nums = [1, 2, 3, 4, 5] | id(nums) = 1579329237248
alias = [1, 2, 3, 4] | id(alias) = 1579329237440


## Solution 3

There are two different operations here:

1. `nums.append(4)` **mutates** the existing list in place.
   - The identity of `nums` does not change.
   - Since `alias` refers to the same list, it also sees the change.

2. `nums = nums + [5]` creates a **new list** and reassigns `nums` to that new object.
   - The identity of `nums` changes.
   - `alias` still points to the old list.

**Key idea:** Mutation changes an object; re-assignment changes what a variable refers to.


---
# Problem 4 - Tracing aliases through a sequence of assignments

Without running the code first, determine the final values of `x`, `y`, and `z`, and decide which variables refer to the same object at the end.


In [6]:
x = [10, 20]
y = x
z = x[:]

y.append(30)
x = x + [40]
z.append(50)

print('x =', x, '| id(x) =', id(x))
print('y =', y, '| id(y) =', id(y))
print('z =', z, '| id(z) =', id(z))

print('x is y ->', x is y)
print('y is z ->', y is z)
print('x is z ->', x is z)


x = [10, 20, 30, 40] | id(x) = 1579329228672
y = [10, 20, 30] | id(y) = 1579307538304
z = [10, 20, 50] | id(z) = 1579329233600
x is y -> False
y is z -> False
x is z -> False


## Solution 4

Step by step:

- `x = [10, 20]` creates a list.
- `y = x` makes `y` an alias of the same list.
- `z = x[:]` makes a **shallow copy**, so `z` is a different list with the same contents initially.
- `y.append(30)` mutates the shared list referred to by `x` and `y`, so both become `[10, 20, 30]`.
- `x = x + [40]` creates a new list `[10, 20, 30, 40]` and reassigns `x` only.
- `z.append(50)` changes only `z`, so `z` becomes `[10, 20, 50]`.

Final result:
- `x == [10, 20, 30, 40]`
- `y == [10, 20, 30]`
- `z == [10, 20, 50]`
- No two variables refer to the same object at the end.

**Best practice:** When debugging aliasing, trace each operation as either **mutation** or **re-assignment**.


---
# Problem 5 - Nested mutable objects and shallow copies

This problem is designed to catch a subtle but common misunderstanding.


In [7]:
outer1 = [[1, 2], [3, 4]]
outer2 = outer1[:]

outer1[0].append(99)
outer1.append(['new'])

print('outer1 =', outer1)
print('outer2 =', outer2)
print('outer1 is outer2 ->', outer1 is outer2)
print('outer1[0] is outer2[0] ->', outer1[0] is outer2[0])
print('outer1[1] is outer2[1] ->', outer1[1] is outer2[1])


outer1 = [[1, 2, 99], [3, 4], ['new']]
outer2 = [[1, 2, 99], [3, 4]]
outer1 is outer2 -> False
outer1[0] is outer2[0] -> True
outer1[1] is outer2[1] -> True


## Solution 5

`outer2 = outer1[:]` creates a **shallow copy** of the outer list only. The inner lists are still shared.

So:
- `outer1[0].append(99)` mutates the inner list shared by both `outer1` and `outer2`.
- `outer1.append(['new'])` changes only the outer list `outer1`; `outer2` is unaffected by that append.

Typical final state:
- `outer1 == [[1, 2, 99], [3, 4], ['new']]`
- `outer2 == [[1, 2, 99], [3, 4]]`

**Key lesson:** A shallow copy duplicates only one level. Nested mutable objects may still be shared.


---
# Problem 6 - Strings and re-assignment

Strings are immutable. Predict what happens here.


In [8]:
s1 = 'data'
s2 = s1

print('Initially:')
print('s1 =', s1, '| id(s1) =', id(s1))
print('s2 =', s2, '| id(s2) =', id(s2))

s1 += '_science'

print('\nAfter s1 += "_science":')
print('s1 =', s1, '| id(s1) =', id(s1))
print('s2 =', s2, '| id(s2) =', id(s2))
print('s1 is s2 ->', s1 is s2)


Initially:
s1 = data | id(s1) = 140722635082440
s2 = data | id(s2) = 140722635082440

After s1 += "_science":
s1 = data_science | id(s1) = 1579329172144
s2 = data | id(s2) = 140722635082440
s1 is s2 -> False


## Solution 6

Because strings are immutable, `s1 += '_science'` does not modify the original string object. Instead, it creates a new string and reassigns `s1` to it.

Therefore:
- `s2` still refers to the original string `'data'`.
- `s1` now refers to `'data_science'`.
- Their identities differ after the operation.

**Key idea:** For immutable objects, augmented assignment often behaves like re-assignment.


---
# Problem 7 - Tuples containing mutable objects

Tuples are immutable, but what if a tuple contains a mutable object?


In [9]:
t = ([1, 2], 'A')
print('Before:', t, '| id(t) =', id(t), '| id(t[0]) =', id(t[0]))

t[0].append(3)
print('After append:', t, '| id(t) =', id(t), '| id(t[0]) =', id(t[0]))


Before: ([1, 2], 'A') | id(t) = 1579329158848 | id(t[0]) = 1579329224896
After append: ([1, 2, 3], 'A') | id(t) = 1579329158848 | id(t[0]) = 1579329224896


## Solution 7

The tuple object itself is immutable, meaning its elements cannot be replaced through assignment like `t[0] = ...`.

However, the first element of the tuple is a **list**, and that list is mutable. So `t[0].append(3)` is allowed because it mutates the list object stored inside the tuple.

- The identity of `t` stays the same.
- The identity of `t[0]` also stays the same.
- The contents of the inner list change.

**Important distinction:** Immutability of a container does not automatically make its contents immutable.


---
# Problem 8 - Function arguments, mutation, and re-binding

Predict the output carefully.


In [10]:
def process(values):
    print('Inside function (start):', values, '| id =', id(values))
    values.append(100)
    print('After append:', values, '| id =', id(values))
    values = values + [200]
    print('After rebinding:', values, '| id =', id(values))

nums = [1, 2, 3]
print('Before function call:', nums, '| id =', id(nums))
process(nums)
print('After function call:', nums, '| id =', id(nums))


Before function call: [1, 2, 3] | id = 1579329232000
Inside function (start): [1, 2, 3] | id = 1579329232000
After append: [1, 2, 3, 100] | id = 1579329232000
After rebinding: [1, 2, 3, 100, 200] | id = 1579329237248
After function call: [1, 2, 3, 100] | id = 1579329232000


## Solution 8

Inside the function, the parameter `values` initially refers to the same list as `nums`.

- `values.append(100)` mutates that shared list, so the caller sees the change.
- `values = values + [200]` creates a new list and rebinds the local name `values` only. This does not affect the caller's variable `nums`.

So after the function call, `nums` becomes:

```python
[1, 2, 3, 100]
```

but it does **not** include `200`.

**Best practice:** Be explicit when functions mutate inputs. Unexpected mutation is a common source of bugs.


---
# Problem 9 - Diagnose the bug

The following code is intended to build independent rows, but it produces a surprising result. Explain why and fix it.


In [11]:
rows = [[0] * 3] * 4
rows[0][1] = 99
print(rows)


[[0, 99, 0], [0, 99, 0], [0, 99, 0], [0, 99, 0]]


## Solution 9

The expression `[[0] * 3] * 4` does **not** create four independent inner lists. It creates one inner list and repeats references to that same list four times.

That means all rows alias the same object, so mutating one row affects them all.

A correct version is:


In [12]:
rows = [[0] * 3 for _ in range(4)]
rows[0][1] = 99
print(rows)


[[0, 99, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]


This list comprehension creates a **new inner list** on each iteration, so the rows are independent.

**Best practice:** Avoid using list multiplication to create nested mutable structures.


---
# Problem 10 - Advanced tracing challenge

This final problem combines identity, copying, mutation, and re-assignment. Work through it carefully.


In [13]:
a = [1, 2]
b = a
c = a.copy()

a.append(3)
b = b + [4]
c.append(5)
a[0] = 99

print('a =', a, '| id(a) =', id(a))
print('b =', b, '| id(b) =', id(b))
print('c =', c, '| id(c) =', id(c))

print('a is b ->', a is b)
print('a is c ->', a is c)
print('b is c ->', b is c)


a = [99, 2, 3] | id(a) = 1579329199808
b = [1, 2, 3, 4] | id(b) = 1579329234880
c = [1, 2, 5] | id(c) = 1579329206656
a is b -> False
a is c -> False
b is c -> False


## Solution 10

Trace it step by step:

1. `a = [1, 2]`
2. `b = a` -> `b` aliases `a`
3. `c = a.copy()` -> `c` is a new list `[1, 2]`
4. `a.append(3)` mutates the shared list used by `a` and `b` -> both see `[1, 2, 3]`
5. `b = b + [4]` creates a new list `[1, 2, 3, 4]` and reassigns `b` only
6. `c.append(5)` makes `c` become `[1, 2, 5]`
7. `a[0] = 99` mutates `a`'s list -> `a` becomes `[99, 2, 3]`

Final values:
- `a == [99, 2, 3]`
- `b == [1, 2, 3, 4]`
- `c == [1, 2, 5]`

All three are different objects at the end.

**Master takeaway:** To reason accurately about Python variables, always ask two questions:
1. Did this line **mutate an existing object**?
2. Or did it **rebind a name** to a different object?


---
# Summary

## Core rules
- Variables are **references (names)** bound to objects.
- `id(obj)` helps inspect object identity.
- Immutable objects cannot be changed in place; operations usually create new objects.
- Mutable objects can often be changed in place.
- `x = y` does not copy an object; it creates another reference to the same object.
- Shallow copies duplicate the outer container but may still share nested mutable objects.

## Best practices
- Use `==` for value comparison, not `is`.
- Use `is` mainly for singleton checks like `x is None`.
- Be careful with aliasing when working with lists, dictionaries, and nested data structures.
- Avoid surprising side effects by documenting or avoiding mutation in functions unless intended.
- When in doubt, print both the value and `id(...)` while debugging.

## Optional extension
Try repeating these experiments with dictionaries, sets, tuples, NumPy arrays, and custom classes to deepen your understanding of identity and mutation.
